# RoBERTa Base SQuAD2 – Extractive Question Answering

**Model:** `deepset/roberta-base-squad2`  
**Task:** Extractive Question Answering  
**Benchmark:** SQuAD 2.0 – F1 82.9, EM 79.9  
**License:** CC-BY-4.0 *(attribution required – see note below)*  
**Marketplace price:** \$0.08/hr

> **Attribution notice (CC-BY-4.0):** When using or redistributing outputs from
> this model, you must credit the original work: *deepset/roberta-base-squad2*
> by deepset GmbH, licensed under CC-BY-4.0
> (<https://creativecommons.org/licenses/by/4.0/>).

This notebook shows how to:
1. Deploy the model endpoint from AWS Marketplace
2. Send extractive QA requests (`question` + `context`)
3. Inspect answer spans with start/end character offsets and confidence scores
4. Handle **unanswerable questions** (SQuAD 2.0 capability)

## Prerequisites

- AWS account with SageMaker execution role that has `AmazonSageMakerFullAccess`
- The model must be subscribed to in AWS Marketplace before deploying
- `boto3` and `sagemaker` Python packages installed

In [ ]:
import boto3
import sagemaker
import json
import time

sess = sagemaker.Session()
try:
    role = sagemaker.get_execution_role()
except ValueError:
    # Running outside SageMaker -- specify your role ARN explicitly
    role = "arn:aws:iam::<ACCOUNT-ID>:role/<SAGEMAKER-ROLE-NAME>"
print("Region:", session.boto_region_name)
region = sess.boto_region_name

print(f"Region : {region}")
print(f"Role   : {role}")

## 1. Deploy the Endpoint

Retrieve the model package ARN from your AWS Marketplace subscription and deploy
it as a real-time SageMaker endpoint.

In [ ]:
# Replace with the model package ARN from your AWS Marketplace subscription
MODEL_PACKAGE_ARN = "<YOUR_MODEL_PACKAGE_ARN>"

ENDPOINT_NAME = "roberta-base-squad2-endpoint"
INSTANCE_TYPE = "ml.g4dn.xlarge"
INSTANCE_COUNT = 1

sm_client = boto3.client("sagemaker", region_name=region)

model_name = f"roberta-squad2-{int(time.time())}"
try:
    sm_client.create_model(
        ModelName=model_name,
        ExecutionRoleArn=role,
        PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    )
except Exception as e:
    print(f"Error in create_model: {e}")
    raise

config_name = f"{model_name}-config"
try:
    sm_client.create_endpoint_config(
        EndpointConfigName=config_name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": INSTANCE_TYPE,
                "InitialInstanceCount": INSTANCE_COUNT,
            }
        ],
    )
except Exception as e:
    print(f"Error in create_endpoint_config: {e}")
    raise

try:
    sm_client.create_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=config_name,
    )
except Exception as e:
    print(f"Error in create_endpoint: {e}")
    raise

print(f"Endpoint '{ENDPOINT_NAME}' creation started. Waiting for InService state...")

In [ ]:
waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=ENDPOINT_NAME,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Endpoint '{ENDPOINT_NAME}' is InService and ready for inference.")

## 2. Extractive QA – Basic Example

The model accepts a JSON payload with:
- `question` – the question to answer
- `context` – the passage to extract the answer from

It returns:
- `answer` – the extracted text span
- `score` – confidence score (0–1)
- `start` – character offset of the answer start in `context`
- `end` – character offset of the answer end in `context`

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)
def answer_question(
    question: str,
    context: str,
    endpoint_name: str = ENDPOINT_NAME,
) -> dict:
    """Run extractive QA. Returns answer, score, and character offsets."""
    payload = json.dumps({"question": question, "context": context})
    try:
    try:
        response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=payload,
        )
        return json.loads(response["Body"].read().decode("utf-8"))
    except Exception as e:
        print(f"Error invoking endpoint: {e}")
        raise
def display_answer(question: str, context: str, result: dict) -> None:
    """Pretty-print QA result with highlighted span."""
    answer = result.get("answer", "")
    start = result.get("start", 0)
    end = result.get("end", 0)
    score = result.get("score", 0.0)
    print(f"Question : {question}")
    print(f"Answer   : {answer!r}")
    print(f"Score    : {score:.4f}")
    print(f"Span     : [{start}:{end}]")
    # Verify the span against the context
    extracted = context[start:end]
    match = "✓" if extracted == answer else "✗ mismatch"
    print(f"Verify   : context[{start}:{end}] = {extracted!r} {match}")
    print()
# --- Example ---
context = """
The Amazon River is the largest river by discharge volume of water in the world
and by some definitions it is the longest. The headwaters of the Apurimac River
on Nevado Mismi had been considered for many years to be the Amazon's most
distant source, but recent studies have shown that the Mantaro River headwaters
in Peru are slightly farther from the Amazon's mouth. The Amazon basin is the
world's largest drainage basin, with an area of approximately 7,000,000 km².
The portion of the river's drainage basin in Brazil alone is larger than any
other river's basin.
""".strip()
question = "What is the area of the Amazon basin?"
result = answer_question(question, context)
display_answer(question, context, result)

## 3. Inspecting Character Offsets

The `start` and `end` fields are **character offsets** into the `context` string.
You can use them to highlight the answer in a UI or extract surrounding context
for display.

In [ ]:
def highlight_answer(context: str, start: int, end: int, window: int = 40) -> str:
    """
    Return the answer span highlighted within its surrounding context window.
    Uses ANSI escape codes for terminal rendering.
    """
    ctx_start = max(0, start - window)
    ctx_end = min(len(context), end + window)
    before = context[ctx_start:start]
    answer = context[start:end]
    after = context[end:ctx_end]
    prefix = "..." if ctx_start > 0 else ""
    suffix = "..." if ctx_end < len(context) else ""
    return f"{prefix}{before}[[[{answer}]]]{after}{suffix}"


questions = [
    "What is the Amazon?",
    "Where are the headwaters of the Apurimac River?",
    "Which country contains the largest portion of the Amazon drainage basin?",
]

for q in questions:
    res = answer_question(q, context)
    print(f"Q: {q}")
    print(f"A: {res['answer']!r}  (score={res['score']:.4f}, span=[{res['start']}:{res['end']}])")
    print(f"   {highlight_answer(context, res['start'], res['end'])}")
    print()

## 4. Unanswerable Questions (SQuAD 2.0 Capability)

Unlike SQuAD 1.1, **SQuAD 2.0** includes questions that have **no answer** in
the context. `deepset/roberta-base-squad2` was fine-tuned on SQuAD 2.0, so it
can abstain from answering.

When the model cannot find a valid answer span, it returns an empty string
(`answer: ""`) with a very low score. You can use a **score threshold** to
detect unanswerable questions and handle them appropriately.

In [ ]:
UNANSWERABLE_THRESHOLD = 0.05  # scores below this are treated as 'no answer'

unanswerable_examples = [
    {
        "question": "Who was the first astronaut to walk on the moon?",
        "context": (
            "The Amazon River is the largest river by discharge volume of water in "
            "the world. The Amazon basin covers approximately 7,000,000 km²."
        ),
        "expected": "unanswerable",
    },
    {
        "question": "What is the capital of France?",
        "context": (
            "Germany is a country in central Europe. Its largest city is Berlin, "
            "which is also the capital. Germany has a population of about 84 million."
        ),
        "expected": "unanswerable",
    },
    {
        "question": "What is the capital of Germany?",
        "context": (
            "Germany is a country in central Europe. Its largest city is Berlin, "
            "which is also the capital. Germany has a population of about 84 million."
        ),
        "expected": "answerable",
    },
]

print(f"Score threshold for 'unanswerable': < {UNANSWERABLE_THRESHOLD}\n")
print(f"{'Question':<50} {'Score':>7} {'Decision':<15} {'Expected':<15}")
print("=" * 90)

for ex in unanswerable_examples:
    res = answer_question(ex["question"], ex["context"])
    score = res.get("score", 0.0)
    decision = "unanswerable" if score < UNANSWERABLE_THRESHOLD else f"'{res['answer']}'"
    truncated_q = ex["question"][:47] + "..." if len(ex["question"]) > 50 else ex["question"]
    print(f"{truncated_q:<50} {score:>7.4f} {decision:<15} {ex['expected']:<15}")

## 5. Document QA – Batch Processing

A practical pattern: run multiple questions over a single document and collect
all answers with their span information.

In [ ]:
document = """
AWS Lambda is a serverless compute service that lets you run code without
provisioning or managing servers. Lambda runs your code only when needed and
scales automatically, from a few requests per day to thousands per second.
You pay only for the compute time you consume. Lambda supports multiple
programming languages including Python, Node.js, Java, Go, and Ruby.
The maximum execution timeout for a Lambda function is 15 minutes.
Lambda functions can be triggered by over 200 AWS services and SaaS applications.
""".strip()

questions = [
    "What is AWS Lambda?",
    "What is the maximum execution timeout for a Lambda function?",
    "Which programming languages does Lambda support?",
    "How many AWS services can trigger Lambda functions?",
]

print(f"Document: {document[:80]}...\n")
print(f"{'#':<3} {'Question':<52} {'Answer':<30} {'Score':>7}")
print("-" * 95)

for i, q in enumerate(questions, 1):
    res = answer_question(q, document)
    answer = res.get("answer", "") or "(no answer)"
    score = res.get("score", 0.0)
    truncated_q = q[:49] + "..." if len(q) > 52 else q
    print(f"{i:<3} {truncated_q:<52} {answer:<30} {score:>7.4f}")

## 6. Clean Up – Delete the Endpoint

Delete the endpoint when you are done to avoid ongoing charges (\$0.08/hr).

In [ ]:
sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm_client.delete_endpoint_config(EndpointConfigName=config_name)
sm_client.delete_model(ModelName=model_name)
print(f"Endpoint '{ENDPOINT_NAME}' and associated resources deleted.")